In [1]:
# !pip install polars pyarrow --quiet

import polars as pl
import numpy as np
from pathlib import Path
import re
import gc
import warnings

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════
DATA_ROOT = Path("/kaggle/input/datasets/nisarggandhi22/aubergine-datasets-allplants")

PLANT_FILES = {
    "plant_1": [
        DATA_ROOT / "Copy of ICR2-LT1-Celestical-10000.73.raws.csv",
        DATA_ROOT / "Copy of ICR2-LT2-Celestical-10000.73.raws.csv",
    ],
    "plant_2": [
        DATA_ROOT / "Copy of 80-1F-12-0F-AC-12.raws.csv",
        DATA_ROOT / "Copy of 80-1F-12-0F-AC-BB.raws.csv",
    ],
    "plant_3": [
        DATA_ROOT / "Copy of 54-10-EC-8C-14-6E.raws.csv",
        DATA_ROOT / "Copy of 54-10-EC-8C-14-69.raws.csv",
    ],
}

OUTPUT_DIR = Path("/kaggle/working/train_ready")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Prediction window
PREDICTION_WINDOW_MIN = 7
PREDICTION_WINDOW_MAX = 10

# Daytime hours (India)
DAYTIME_START = 6
DAYTIME_END = 18

# Metadata columns to always drop
METADATA_DROPS = ["_id", "__v", "mac", "fromServer", "dataLoggerModelId",
                  "createdAt", "grid_master", "batteries"]

# Cleaning
NULL_THRESHOLD = 0.90
MAX_FFILL_GAP = 6

print("✅ Configuration loaded")

✅ Configuration loaded


---
## PHASE 1: Clean, Reshape, Resample
---

### 1.1 Helper Functions

In [2]:
def parse_column_schema(columns: list[str]) -> dict:
    """Parse subsystem[index].field column naming pattern."""
    pattern = re.compile(r"^(\w+)\[(\d+)\]\.(.+)$")
    subsystems = {}
    meta_cols = []
    for col in columns:
        match = pattern.match(col)
        if match:
            subsystem, idx, field = match.group(1), int(match.group(2)), match.group(3)
            subsystems.setdefault(subsystem, {}).setdefault(idx, []).append(field)
        else:
            meta_cols.append(col)
    return {"subsystems": subsystems, "meta": meta_cols}


def unpivot_inverters(df: pl.DataFrame, schema: dict) -> pl.DataFrame:
    """Reshape from wide (all inverters in columns) to long (one row per inverter)."""
    subsystems = schema["subsystems"]
    if "inverters" not in subsystems:
        return df

    inverter_indices = sorted(subsystems["inverters"].keys())
    if len(inverter_indices) <= 1:
        rename_map = {}
        for col in df.columns:
            m = re.match(r"^inverters\[0\]\.(.+)$", col)
            if m:
                rename_map[col] = f"inv_{m.group(1)}"
        if rename_map:
            df = df.rename(rename_map)
            df = df.with_columns(pl.lit(0).cast(pl.Int16).alias("inverter_idx"))
        return df

    # Find common fields
    common_fields = set(subsystems["inverters"][inverter_indices[0]])
    for idx in inverter_indices[1:]:
        common_fields &= set(subsystems["inverters"][idx])

    inverter_col_pattern = re.compile(r"^inverters\[\d+\]\.")
    shared_cols = [c for c in df.columns if not inverter_col_pattern.match(c)]

    inverter_dfs = []
    for idx in inverter_indices:
        inv_cols = {}
        for field in common_fields:
            src = f"inverters[{idx}].{field}"
            if src in df.columns:
                inv_cols[src] = f"inv_{field}"
        if not inv_cols:
            continue
        sub = df.select(shared_cols + list(inv_cols.keys()))
        sub = sub.rename(inv_cols)
        sub = sub.with_columns(pl.lit(idx).cast(pl.Int16).alias("inverter_idx"))
        inverter_dfs.append(sub)

    if inverter_dfs:
        df_long = pl.concat(inverter_dfs, how="diagonal_relaxed")
        print(f"    ✅ Unpivoted: {df.height:,} → {df_long.height:,} rows ({len(inverter_indices)} inverters)")
        return df_long
    return df


def handle_smu(df: pl.DataFrame, schema: dict) -> pl.DataFrame:
    """Map SMU[i] columns to matching inverter_idx rows."""
    subsystems = schema["subsystems"]
    if "smu" not in subsystems:
        return df

    smu_indices = sorted(subsystems["smu"].keys())
    common_smu = set(subsystems["smu"][smu_indices[0]])
    for idx in smu_indices[1:]:
        common_smu &= set(subsystems["smu"][idx])

    existing_smu = [c for c in df.columns if c.startswith("smu[")]
    if not existing_smu or "inverter_idx" not in df.columns:
        # Just rename smu[0].* → smu_*
        rename_map = {}
        for col in df.columns:
            m = re.match(r"^smu\[\d+\]\.(.+)$", col)
            if m:
                target = f"smu_{m.group(1)}"
                if target not in rename_map.values():
                    rename_map[col] = target
        if rename_map:
            df = df.rename(rename_map)
        return df

    for field in common_smu:
        target = f"smu_{field}"
        # Build when/then chain
        expr = pl.lit(None).cast(pl.Float64)
        for idx in smu_indices:
            src = f"smu[{idx}].{field}"
            if src in df.columns:
                expr = pl.when(pl.col("inverter_idx") == idx).then(pl.col(src)).otherwise(expr)
        df = df.with_columns(expr.alias(target))

    smu_originals = [c for c in df.columns if c.startswith("smu[")]
    if smu_originals:
        df = df.drop(smu_originals)
        print(f"    ✅ SMU: mapped {len(common_smu)} fields, dropped {len(smu_originals)} originals")

    return df


def flatten_remaining(df: pl.DataFrame) -> pl.DataFrame:
    """Rename remaining subsystem[idx].field → subsystem_field."""
    rename_map = {}
    for col in df.columns:
        m = re.match(r"^(\w+)\[(\d+)\]\.(.+)$", col)
        if m:
            subsys = m.group(1).rstrip("s") if m.group(1).endswith("s") else m.group(1)
            new_name = f"{subsys}_{m.group(3)}"
            if new_name in rename_map.values() or new_name in df.columns:
                new_name = f"{subsys}{m.group(2)}_{m.group(3)}"
            rename_map[col] = new_name
    if rename_map:
        df = df.rename(rename_map)
    return df

### 1.2 Load, Clean, Reshape — One Plant at a Time

In [3]:
def process_single_csv(filepath: Path) -> pl.DataFrame:
    """Load, clean, and reshape a single CSV file."""
    print(f"\n  📄 {filepath.name}")

    df = pl.read_csv(
        filepath,
        infer_schema_length=10_000,
        ignore_errors=True,
        truncate_ragged_lines=True,
        null_values=["", "NA", "N/A", "null", "NULL", "None", "nan",
                     "NaN", "-", "--", "?", "#N/A", "#VALUE!", "undefined"],
    )
    print(f"    Loaded: {df.height:,} × {df.width}")

    schema = parse_column_schema(df.columns)

    # Drop metadata
    drops = [c for c in df.columns if any(m.lower() in c.lower() for m in METADATA_DROPS)]
    drops = [c for c in drops if c != "timestamp"]
    if drops:
        df = df.drop([c for c in drops if c in df.columns])

    # Drop entirely null and high-null columns
    null_drops = [c for c in df.columns if df[c].null_count() / df.height > NULL_THRESHOLD]
    if null_drops:
        df = df.drop(null_drops)
        print(f"    🗑️  Dropped {len(null_drops)} high-null columns")

    # Parse timestamps
    if "timestamp" in df.columns and df["timestamp"].dtype == pl.String:
        df = df.with_columns(pl.col("timestamp").str.to_datetime(strict=False))
    if "timestampDate" in df.columns:
        if "timestamp" not in df.columns or df["timestamp"].null_count() == df.height:
            if df["timestampDate"].dtype == pl.String:
                df = df.with_columns(pl.col("timestampDate").str.to_datetime(strict=False).alias("timestamp"))
        df = df.drop("timestampDate")

    # Drop zero-variance
    const_cols = [c for c in df.columns
                  if df[c].drop_nulls().n_unique() <= 1
                  and c not in ("timestamp",)]
    if const_cols:
        df = df.drop(const_cols)
        print(f"    🗑️  Dropped {len(const_cols)} zero-variance columns")

    # Drop identifier columns (model, serial, id) 
    id_cols = [c for c in df.columns if re.search(r"\.(model|serial|id)$", c)]
    if id_cols:
        df = df.drop(id_cols)

    # Re-parse schema after drops
    schema = parse_column_schema(df.columns)

    # Unpivot inverters
    df = unpivot_inverters(df, schema)

    # Handle SMU
    df = handle_smu(df, schema)

    # Flatten remaining (meters, sensors)
    df = flatten_remaining(df)

    # Coerce string columns that are actually numeric
    for col_name in df.columns:
        if df[col_name].dtype != pl.String:
            continue
        sample = df[col_name].drop_nulls().head(100)
        if sample.len() == 0:
            continue
        try:
            casted = sample.cast(pl.Float64, strict=False)
            if 1.0 - (casted.null_count() / sample.len()) > 0.8:
                df = df.with_columns(pl.col(col_name).cast(pl.Float64, strict=False))
        except Exception:
            pass

    print(f"    ✅ Clean shape: {df.height:,} × {df.width}")
    return df

In [4]:
def resample_hourly(df: pl.DataFrame) -> pl.DataFrame:
    """
    Resample to hourly. KEY FIX: use mode/max for categorical columns
    (op_state, alarm_code) instead of mean.
    """
    # Fix timestamp if it's not Datetime
    if df["timestamp"].dtype != pl.Datetime:
        if df["timestamp"].dtype in (pl.Float64, pl.Float32, pl.Int64):
            sample_val = df["timestamp"].drop_nulls().head(1).item()
            if sample_val > 1e12:
                df = df.with_columns(
                    pl.from_epoch(pl.col("timestamp").cast(pl.Int64), time_unit="ms").alias("timestamp")
                )
            else:
                df = df.with_columns(
                    pl.from_epoch(pl.col("timestamp").cast(pl.Int64), time_unit="s").alias("timestamp")
                )

    # Categorize columns
    group_cols = {"timestamp", "inverter_idx"}

    # Categorical/status columns — use MODE (most frequent value in the hour)
    categorical_patterns = re.compile(r"(op_state|alarm_code|alarm|fault|status|flag)", re.I)
    categorical_cols = [c for c in df.columns
                        if categorical_patterns.search(c)
                        and c not in group_cols
                        and df[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32,
                                            pl.Int16, pl.Int8, pl.UInt32)]

    # Numeric telemetry columns — use MEAN
    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8, pl.UInt32)
    numeric_cols = [c for c in df.columns
                    if df[c].dtype in numeric_types
                    and c not in group_cols
                    and c not in categorical_cols]

    # Drop non-numeric non-group columns (strings that survived cleaning)
    drop_cols = [c for c in df.columns
                 if c not in group_cols
                 and c not in numeric_cols
                 and c not in categorical_cols
                 and c != "timestamp"]
    if drop_cols:
        df = df.drop(drop_cols)

    df = df.sort(["inverter_idx", "timestamp"])

    # Build aggregation expressions
    agg_exprs = []

    # Categorical: take the mode (most frequent value in the hour)
    # Polars mode() can return multiple values, so we take first
    for c in categorical_cols:
        agg_exprs.append(
            pl.col(c).drop_nulls().mode().first().alias(c)
        )

    # Numeric: take the mean
    for c in numeric_cols:
        agg_exprs.append(pl.col(c).mean().alias(c))

    df = df.group_by_dynamic(
        "timestamp", every="1h", group_by="inverter_idx"
    ).agg(agg_exprs)

    print(f"    ✅ Resampled to hourly: {df.height:,} rows × {df.width} cols")
    return df

In [5]:
def impute_missing(df: pl.DataFrame) -> pl.DataFrame:
    """Time-series aware imputation, per inverter."""
    alarm_pattern = re.compile(r"(alarm|fault|error|warning|flag|status|op_state)", re.I)

    numeric_cols = [c for c in df.columns if df[c].dtype in (
        pl.Float32, pl.Float64, pl.Int64, pl.Int32, pl.Int16, pl.Int8)]

    alarm_cols = [c for c in numeric_cols if alarm_pattern.search(c)]
    telemetry_cols = [c for c in numeric_cols if c not in alarm_cols]

    has_group = "inverter_idx" in df.columns

    if telemetry_cols:
        if has_group:
            df = df.with_columns([
                pl.col(c).forward_fill(limit=MAX_FFILL_GAP).over("inverter_idx")
                for c in telemetry_cols
            ])
            df = df.with_columns([
                pl.col(c).backward_fill(limit=2).over("inverter_idx")
                for c in telemetry_cols
            ])
        else:
            df = df.with_columns([pl.col(c).forward_fill(limit=MAX_FFILL_GAP) for c in telemetry_cols])
            df = df.with_columns([pl.col(c).backward_fill(limit=2) for c in telemetry_cols])

    if alarm_cols:
        df = df.with_columns([pl.col(c).fill_null(0) for c in alarm_cols])

    return df

### 1.3 Run Phase 1 — Process All Plants

In [6]:
plant_dfs = {}

print("=" * 70)
print("PHASE 1: CLEAN → RESHAPE → RESAMPLE")
print("=" * 70)

for plant_name, filepaths in PLANT_FILES.items():
    print(f"\n{'━'*60}")
    print(f"📂 {plant_name.upper()}")
    print(f"{'━'*60}")

    file_dfs = []
    for fp in filepaths:
        if not fp.exists():
            print(f"\n  ❌ {fp.name}: NOT FOUND")
            continue
        df = process_single_csv(fp)
        file_dfs.append(df)
        del df
        gc.collect()

    if not file_dfs:
        continue

    # Merge files within plant
    if len(file_dfs) > 1:
        merged = pl.concat(file_dfs, how="diagonal_relaxed")
        print(f"\n  📦 Merged: {merged.height:,} rows × {merged.width} cols")
    else:
        merged = file_dfs[0]

    del file_dfs
    gc.collect()

    # Resample to hourly
    merged = resample_hourly(merged)

    # Impute missing values
    nulls_before = sum(merged[c].null_count() for c in merged.columns)
    merged = impute_missing(merged)
    nulls_after = sum(merged[c].null_count() for c in merged.columns)
    print(f"    Nulls: {nulls_before:,} → {nulls_after:,}")

    plant_dfs[plant_name] = merged
    del merged
    gc.collect()

print("\n✅ Phase 1 complete")
for name, df in plant_dfs.items():
    print(f"  {name}: {df.height:,} rows × {df.width} cols")

PHASE 1: CLEAN → RESHAPE → RESAMPLE

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 PLANT_1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  📄 Copy of ICR2-LT1-Celestical-10000.73.raws.csv
    Loaded: 189,421 × 442
    🗑️  Dropped 25 zero-variance columns
    ✅ Unpivoted: 189,421 → 2,273,052 rows (12 inverters)
    ✅ SMU: mapped 24 fields, dropped 288 originals
    ✅ Clean shape: 2,273,052 × 36

  📄 Copy of ICR2-LT2-Celestical-10000.73.raws.csv
    Loaded: 189,213 × 406
    🗑️  Dropped 23 zero-variance columns
    ✅ Unpivoted: 189,213 → 2,081,343 rows (11 inverters)
    ✅ SMU: mapped 24 fields, dropped 264 originals
    ✅ Clean shape: 2,081,343 × 36

  📦 Merged: 4,354,395 rows × 36 cols
    ✅ Resampled to hourly: 181,958 rows × 36 cols
    Nulls: 6,429 → 0

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 PLANT_2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  📄 Copy of 80-1F-12-0F-AC-12.raws.csv
    Loaded: 190,899 × 132
   

### 1.4 Verify op_state and alarm_code are intact

In [7]:
print("=" * 70)
print("VERIFICATION: op_state & alarm_code values")
print("=" * 70)

for plant_name, df in plant_dfs.items():
    print(f"\n📂 {plant_name}")

    if "inv_op_state" in df.columns:
        vc = df.group_by("inv_op_state").agg(pl.len().alias("count")).sort("count", descending=True).head(10)
        print(f"  inv_op_state (top 10):\n{vc}")
    else:
        print("  ⚠️  No inv_op_state")

    if "inv_alarm_code" in df.columns:
        vc = df.group_by("inv_alarm_code").agg(pl.len().alias("count")).sort("count", descending=True).head(10)
        print(f"  inv_alarm_code (top 10):\n{vc}")
    else:
        print("  ⚠️  No inv_alarm_code")

    if "timestamp" in df.columns:
        print(f"  Timestamp range: {df['timestamp'].min()} → {df['timestamp'].max()}")

    if "inverter_idx" in df.columns:
        print(f"  Inverters: {df['inverter_idx'].n_unique()}")

VERIFICATION: op_state & alarm_code values

📂 plant_1
  inv_op_state (top 10):
shape: (2, 2)
┌──────────────┬───────┐
│ inv_op_state ┆ count │
│ ---          ┆ ---   │
│ f64          ┆ u32   │
╞══════════════╪═══════╡
│ -1.0         ┆ 96142 │
│ 0.0          ┆ 85816 │
└──────────────┴───────┘
  ⚠️  No inv_alarm_code
  Timestamp range: 2024-03-01 04:00:00 → 2026-03-02 11:00:00
  Inverters: 12

📂 plant_2
  inv_op_state (top 10):
shape: (9, 2)
┌──────────────┬───────┐
│ inv_op_state ┆ count │
│ ---          ┆ ---   │
│ i64          ┆ u32   │
╞══════════════╪═══════╡
│ 0            ┆ 39529 │
│ 5120         ┆ 37373 │
│ 37120        ┆ 2929  │
│ 4864         ┆ 186   │
│ 5632         ┆ 130   │
│ 21760        ┆ 9     │
│ 33280        ┆ 7     │
│ 33024        ┆ 2     │
│ 4608         ┆ 1     │
└──────────────┴───────┘
  inv_alarm_code (top 10):
shape: (10, 2)
┌────────────────┬───────┐
│ inv_alarm_code ┆ count │
│ ---            ┆ ---   │
│ i64            ┆ u32   │
╞════════════════╪═══════╡
│ 0 

---
## PHASE 2: Target Variable & Advanced Features
---

### 2.1 Create Target Variable

In [9]:
def identify_failure_events(df: pl.DataFrame, plant_name: str) -> pl.DataFrame:
    """
    Plant-specific failure event detection.
    Uses EXACT values since op_state was resampled with mode(), not mean().

    Inverter op_state meanings (typical):
    - Plant 1: -1 = fault/shutdown, 0 = off/standby, positive = running
    - Plant 2: 5120 = running, 0 = off, high values (37120+) = error states
    - Plant 3: 4 = running (MPP), 0 = off
    """
    # Add hour column
    if "hour" not in df.columns and "timestamp" in df.columns:
        if df["timestamp"].dtype in (pl.Datetime,):
            df = df.with_columns(pl.col("timestamp").dt.hour().alias("hour"))

    daytime = (pl.col("hour") >= DAYTIME_START) & (pl.col("hour") <= DAYTIME_END)

    if plant_name == "plant_1":
        # op_state == -1 during daytime = fault/shutdown
        # op_state == 0 during daytime could also be a shutdown (inverter should be producing)
        if "inv_op_state" in df.columns:
            df = df.with_columns(
                (
                    (pl.col("inv_op_state") < 0) & daytime
                ).cast(pl.Int8).alias("is_failure_event")
            )
        else:
            df = df.with_columns(pl.lit(0).cast(pl.Int8).alias("is_failure_event"))

    elif plant_name == "plant_2":
        conditions = []

        if "inv_op_state" in df.columns:
            # Daytime shutdown: op_state == 0 during day
            conditions.append((pl.col("inv_op_state") == 0) & daytime)
            # Error states: very high op_state values (bit flags indicating errors)
            conditions.append(pl.col("inv_op_state") > 10000)

        if "inv_alarm_code" in df.columns:
            # Non-zero alarm code = alarm active
            conditions.append(pl.col("inv_alarm_code") > 0)

        if conditions:
            combined = conditions[0]
            for c in conditions[1:]:
                combined = combined | c
            df = df.with_columns(combined.cast(pl.Int8).alias("is_failure_event"))
        else:
            df = df.with_columns(pl.lit(0).cast(pl.Int8).alias("is_failure_event"))

    elif plant_name == "plant_3":
        conditions = []

        if "inv_op_state" in df.columns:
            # op_state 0 during daytime = not running when it should be
            conditions.append((pl.col("inv_op_state") == 0) & daytime)

        if "inv_alarm_code" in df.columns:
            # Non-zero alarm with low op_state = failure
            # alarm_code 100 is very common (41%) — likely a persistent warning
            # Still include it but note this for analysis
            conditions.append(
                (pl.col("inv_alarm_code") > 0) & (pl.col("inv_op_state") < 2) & daytime
            )

        if conditions:
            combined = conditions[0]
            for c in conditions[1:]:
                combined = combined | c
            df = df.with_columns(combined.cast(pl.Int8).alias("is_failure_event"))
        else:
            df = df.with_columns(pl.lit(0).cast(pl.Int8).alias("is_failure_event"))

    else:
        df = df.with_columns(pl.lit(0).cast(pl.Int8).alias("is_failure_event"))

    n_fail = df["is_failure_event"].sum()
    pct = n_fail / df.height * 100
    print(f"    Failure events: {n_fail:,} / {df.height:,} ({pct:.2f}%)")
    return df


def create_forward_looking_target(df: pl.DataFrame) -> pl.DataFrame:
    """
    For each row, check if the same inverter has a failure event
    within the next 7–10 days. target=1 if yes.
    """
    ts_col = "timestamp"
    group_col = "inverter_idx"
    has_group = group_col in df.columns

    if ts_col not in df.columns or df[ts_col].dtype not in (pl.Datetime,):
        print("    ❌ No valid timestamp — using is_failure_event as target")
        df = df.with_columns(pl.col("is_failure_event").alias("target"))
        return df

    sort_cols = ([group_col] if has_group else []) + [ts_col]
    df = df.sort(sort_cols)

    # Mark failure timestamps
    df = df.with_columns(
        pl.when(pl.col("is_failure_event") == 1)
        .then(pl.col(ts_col))
        .otherwise(None)
        .alias("_failure_ts")
    )

    # Backward fill: each row gets the timestamp of its NEXT failure
    if has_group:
        df = df.with_columns(
            pl.col("_failure_ts").backward_fill().over(group_col).alias("_next_failure_ts")
        )
    else:
        df = df.with_columns(
            pl.col("_failure_ts").backward_fill().alias("_next_failure_ts")
        )

    # Days until next failure
    df = df.with_columns(
        (pl.col("_next_failure_ts") - pl.col(ts_col))
        .dt.total_seconds().truediv(86400)
        .alias("days_to_next_failure")
    )

    # Target: failure within 7–10 day window
    df = df.with_columns(
        pl.when(
            (pl.col("days_to_next_failure") >= PREDICTION_WINDOW_MIN) &
            (pl.col("days_to_next_failure") <= PREDICTION_WINDOW_MAX)
        ).then(1).otherwise(0)
        .cast(pl.Int8).alias("target")
    )

    # Broader target: failure within 0–10 days (for comparison)
    df = df.with_columns(
        pl.when(
            (pl.col("days_to_next_failure") >= 0) &
            (pl.col("days_to_next_failure") <= PREDICTION_WINDOW_MAX)
        ).then(1).otherwise(0)
        .cast(pl.Int8).alias("target_any_within_10d")
    )

    df = df.drop(["_failure_ts", "_next_failure_ts"])

    n_pos = df["target"].sum()
    pct = n_pos / df.height * 100 if df.height > 0 else 0
    print(f"    Target (7–10d): {n_pos:,} positive ({pct:.2f}%)")
    n_pos2 = df["target_any_within_10d"].sum()
    pct2 = n_pos2 / df.height * 100 if df.height > 0 else 0
    print(f"    Target (0–10d): {n_pos2:,} positive ({pct2:.2f}%)")

    return df

In [11]:
def identify_failure_events(df: pl.DataFrame, plant_name: str) -> pl.DataFrame:
    """
    Universal failure definition: daytime zero-power output.
    This is the actual business impact — inverter not producing when it should be.
    """
    if "hour" not in df.columns and "timestamp" in df.columns:
        if df["timestamp"].dtype in (pl.Datetime,):
            df = df.with_columns(pl.col("timestamp").dt.hour().alias("hour"))

    daytime = (pl.col("hour") >= DAYTIME_START) & (pl.col("hour") <= DAYTIME_END)

    if "inv_power" in df.columns:
        df = df.with_columns(
            ((pl.col("inv_power").abs() < 0.1) & daytime)
            .cast(pl.Int8).alias("is_failure_event")
        )
    else:
        df = df.with_columns(pl.lit(0).cast(pl.Int8).alias("is_failure_event"))

    n_fail = df["is_failure_event"].sum()
    pct = n_fail / df.height * 100
    print(f"    Failure events: {n_fail:,} / {df.height:,} ({pct:.2f}%)")
    return df


# Re-run target creation
print("=" * 70)
print("RE-CREATING TARGET VARIABLE")
print("=" * 70)

for plant_name, df in plant_dfs.items():
    print(f"\n📂 {plant_name}")

    # Remove old target columns
    drop_cols = [c for c in ["is_failure_event", "target", "target_any_within_10d", 
                              "days_to_next_failure"] if c in df.columns]
    if drop_cols:
        df = df.drop(drop_cols)

    df = identify_failure_events(df, plant_name)
    df = create_forward_looking_target(df)
    plant_dfs[plant_name] = df

# Quick check
for plant_name, df in plant_dfs.items():
    n_pos = df["target"].sum()
    n_fail = df["is_failure_event"].sum()
    n_pos_10d = df["target_any_within_10d"].sum()
    ratio = (df.height - n_pos) / max(n_pos, 1)
    print(f"📂 {plant_name}: failures={n_fail:,}  target(7-10d)={n_pos:,} (1:{ratio:.0f})  target(0-10d)={n_pos_10d:,}  total={df.height:,}")

RE-CREATING TARGET VARIABLE

📂 plant_1
    Failure events: 3,087 / 181,958 (1.70%)
    Target (7–10d): 11,699 positive (6.43%)
    Target (0–10d): 119,505 positive (65.68%)

📂 plant_2
    Failure events: 2,631 / 80,166 (3.28%)
    Target (7–10d): 5,013 positive (6.25%)
    Target (0–10d): 64,230 positive (80.12%)

📂 plant_3
    Failure events: 528 / 16,501 (3.20%)
    Target (7–10d): 626 positive (3.79%)
    Target (0–10d): 12,377 positive (75.01%)
📂 plant_1: failures=3,087  target(7-10d)=11,699 (1:15)  target(0-10d)=119,505  total=181,958
📂 plant_2: failures=2,631  target(7-10d)=5,013 (1:15)  target(0-10d)=64,230  total=80,166
📂 plant_3: failures=528  target(7-10d)=626 (1:25)  target(0-10d)=12,377  total=16,501


### 2.2 Advanced Domain-Specific Features

In [12]:
print("=" * 70)
print("PHASE 2.2: ADVANCED FEATURE ENGINEERING")
print("=" * 70)

for plant_name, df in plant_dfs.items():
    print(f"\n{'━'*60}")
    print(f"📂 {plant_name} — {df.height:,} × {df.width}")
    print(f"{'━'*60}")

    # ── Global cleanup: drop null timestamps, sort ──
    null_ts = df["timestamp"].null_count()
    if null_ts > 0:
        df = df.filter(pl.col("timestamp").is_not_null())
        print(f"    ⚠️  Dropped {null_ts} null-timestamp rows")
    df = df.sort(["inverter_idx", "timestamp"] if "inverter_idx" in df.columns else ["timestamp"])

    w0 = df.width
    
def add_temporal_features(df: pl.DataFrame) -> pl.DataFrame:
    """Time-based features."""
    if "timestamp" not in df.columns or df["timestamp"].dtype not in (pl.Datetime,):
        return df
    if "hour" not in df.columns:
        df = df.with_columns(pl.col("timestamp").dt.hour().alias("hour"))

    df = df.with_columns([
        pl.col("timestamp").dt.weekday().alias("day_of_week"),
        pl.col("timestamp").dt.month().alias("month"),
        ((pl.col("hour") >= DAYTIME_START) & (pl.col("hour") <= DAYTIME_END))
        .cast(pl.Int8).alias("is_daytime"),
    ])
    return df

def add_rolling_telemetry_features(df: pl.DataFrame) -> pl.DataFrame:
    """Rolling mean and std for key telemetry signals (row-based, 1h per row)."""
    has_group = "inverter_idx" in df.columns

    key_signals = [s for s in ["inv_power", "inv_temp", "inv_freq", "inv_pv1_power",
                                "inv_kwh_today", "inv_v_ab", "inv_v_bc", "inv_v_ca"]
                   if s in df.columns]
    if not key_signals:
        return df

    # Row-based windows: 24=1d, 72=3d, 168=7d (hourly data)
    windows = {"3d": 72, "7d": 168}
    added = 0

    for signal in key_signals:
        sig_safe = signal.replace("inv_", "")
        for label, w in windows.items():
            mean_expr = pl.col(signal).rolling_mean(window_size=w, min_periods=1)
            std_expr = pl.col(signal).rolling_std(window_size=w, min_periods=2)
            if has_group:
                mean_expr = mean_expr.over("inverter_idx")
                std_expr = std_expr.over("inverter_idx")
            try:
                df = df.with_columns([
                    mean_expr.alias(f"roll_{sig_safe}_mean_{label}"),
                    std_expr.alias(f"roll_{sig_safe}_std_{label}"),
                ])
                added += 2
            except Exception as e:
                print(f"      ⚠️  Rolling {signal} {label} failed: {e}")

    print(f"    ✅ Rolling telemetry: {added} features")
    return df


def add_degradation_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compare current performance against inverter's own historical baseline."""
    has_group = "inverter_idx" in df.columns
    new_cols = []
    temp_cols = []

    windows = {"7d": 168, "14d": 336}

    for signal, label in [("inv_power", "power"), ("inv_kwh_today", "kwh"), ("inv_temp", "temp")]:
        if signal not in df.columns:
            continue
        for wlabel, wsize in windows.items():
            rmean_col = f"_{label}_rmean_{wlabel}"
            expr = pl.col(signal).rolling_mean(window_size=wsize, min_periods=1)
            if has_group:
                expr = expr.over("inverter_idx")
            try:
                df = df.with_columns(expr.alias(rmean_col))
                temp_cols.append(rmean_col)

                if label == "temp":
                    new_cols.append((pl.col(signal) - pl.col(rmean_col)).alias(f"degrad_{label}_dev_{wlabel}"))
                else:
                    new_cols.append(
                        (pl.col(signal) / (pl.col(rmean_col) + 1e-6)).alias(f"degrad_{label}_ratio_{wlabel}")
                    )
            except Exception as e:
                print(f"      ⚠️  Degradation {signal} {wlabel} failed: {e}")

    if new_cols:
        df = df.with_columns(new_cols)
        print(f"    ✅ Degradation: {len(new_cols)} features")

    temp_cols = [c for c in temp_cols if c in df.columns]
    if temp_cols:
        df = df.drop(temp_cols)

    return df


def add_cumulative_stress_features(df: pl.DataFrame) -> pl.DataFrame:
    """Accumulated stress indicators."""
    has_group = "inverter_idx" in df.columns
    added = 0

    # Cumulative high-temp hours in last 7 days (168 rows)
    if "inv_temp" in df.columns:
        df = df.with_columns((pl.col("inv_temp") > 55).cast(pl.Int8).alias("_temp_stress"))
        expr = pl.col("_temp_stress").rolling_sum(window_size=168, min_periods=1)
        if has_group:
            expr = expr.over("inverter_idx")
        df = df.with_columns(expr.alias("stress_hightemp_7d"))
        df = df.drop("_temp_stress")
        added += 1

    # Daytime zero-power: rolling count in last 24h and 72h
    if "inv_power" in df.columns and "hour" in df.columns:
        df = df.with_columns(
            ((pl.col("inv_power").abs() < 0.1) &
             (pl.col("hour") >= DAYTIME_START) &
             (pl.col("hour") <= DAYTIME_END))
            .cast(pl.Int8).alias("_dz")
        )

        for label, w in {"1d": 24, "3d": 72}.items():
            expr = pl.col("_dz").rolling_sum(window_size=w, min_periods=1)
            if has_group:
                expr = expr.over("inverter_idx")
            try:
                df = df.with_columns(expr.alias(f"stress_zero_power_{label}"))
                added += 1
            except Exception:
                pass

        df = df.drop("_dz")

    # Rolling failure event count
    if "is_failure_event" in df.columns:
        for label, w in {"3d": 72, "7d": 168}.items():
            expr = pl.col("is_failure_event").rolling_sum(window_size=w, min_periods=1)
            if has_group:
                expr = expr.over("inverter_idx")
            try:
                df = df.with_columns(expr.alias(f"stress_failure_count_{label}"))
                added += 1
            except Exception:
                pass

    print(f"    ✅ Cumulative stress: {added} features")
    return df




def add_inter_inverter_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compare each inverter against plant-level peers at same timestamp."""
    if "inverter_idx" not in df.columns or df["inverter_idx"].n_unique() < 2:
        print("    ⚠️  <2 inverters — skipping peer comparison")
        return df

    n_inv = df["inverter_idx"].n_unique()
    new_cols = []
    temp_cols = []

    for signal, name in [("inv_power", "power"), ("inv_temp", "temp")]:
        if signal not in df.columns:
            continue

        med = f"_plant_med_{name}"
        std = f"_plant_std_{name}"
        df = df.with_columns([
            pl.col(signal).median().over("timestamp").alias(med),
            pl.col(signal).std().over("timestamp").alias(std),
        ])
        temp_cols.extend([med, std])

        new_cols.append((pl.col(signal) - pl.col(med)).alias(f"peer_{name}_dev"))
        new_cols.append(
            ((pl.col(signal) - pl.col(med)) / (pl.col(std) + 1e-6)).alias(f"peer_{name}_zscore")
        )

        if name == "power":
            new_cols.append(pl.col(signal).rank("ordinal").over("timestamp").alias(f"peer_{name}_rank"))

    # Ratio vs best inverter
    if "inv_power" in df.columns:
        df = df.with_columns(pl.col("inv_power").max().over("timestamp").alias("_pmax"))
        temp_cols.append("_pmax")
        new_cols.append((pl.col("inv_power") / (pl.col("_pmax") + 1e-6)).alias("peer_power_vs_best"))

    if new_cols:
        df = df.with_columns(new_cols)
        print(f"    ✅ Inter-inverter: {len(new_cols)} features ({n_inv} inverters)")

    # Drop intermediate columns AFTER they've been used
    temp_cols = [c for c in temp_cols if c in df.columns]
    if temp_cols:
        df = df.drop(temp_cols)

    return df




def add_nighttime_anomaly_features(df: pl.DataFrame) -> pl.DataFrame:
    """Detect anomalies during nighttime when inverters should be off."""
    if "hour" not in df.columns:
        return df

    has_group = "inverter_idx" in df.columns
    night = (pl.col("hour") < DAYTIME_START) | (pl.col("hour") > DAYTIME_END)
    new_cols = []

    if "inv_power" in df.columns:
        new_cols.append(
            pl.when(night).then((pl.col("inv_power").abs() > 0.5).cast(pl.Int8))
            .otherwise(0).alias("anom_night_power")
        )
    if "inv_temp" in df.columns:
        new_cols.append(
            pl.when(night).then((pl.col("inv_temp") > 40).cast(pl.Int8))
            .otherwise(0).alias("anom_night_hightemp")
        )

    if new_cols:
        df = df.with_columns(new_cols)
        # Rolling count
        for ac in ["anom_night_power", "anom_night_hightemp"]:
            if ac in df.columns:
                expr = pl.col(ac).rolling_sum(window_size=168, min_periods=1)
                if has_group:
                    expr = expr.over("inverter_idx")
                try:
                    df = df.with_columns(expr.alias(f"{ac}_7d"))
                except Exception:
                    pass
        print(f"    ✅ Nighttime anomaly: {len(new_cols)} base + rolling")
    return df


def add_string_health_features(df: pl.DataFrame) -> pl.DataFrame:
    """Analyze SMU string-level currents for degradation."""
    has_group = "inverter_idx" in df.columns
    has_ts = "timestamp" in df.columns and df["timestamp"].dtype in (pl.Datetime,)

    smu_cols = sorted([c for c in df.columns if re.match(r"smu_string\d+$", c)])
    if len(smu_cols) < 3:
        print(f"    ⚠️  {len(smu_cols)} SMU cols — skipping string health")
        return df

    print(f"    Analyzing {len(smu_cols)} SMU strings...")

    df = df.with_columns([
        pl.mean_horizontal(smu_cols).alias("str_mean"),
        pl.min_horizontal(smu_cols).alias("str_min"),
        pl.max_horizontal(smu_cols).alias("str_max"),
        (pl.max_horizontal(smu_cols) - pl.min_horizontal(smu_cols)).alias("str_spread"),
        (pl.min_horizontal(smu_cols) / (pl.max_horizontal(smu_cols) + 1e-6)).alias("str_worst_ratio"),
    ])

    # Count underperforming strings (< 50% of mean)
    row_mean = pl.mean_horizontal(smu_cols)
    underperf = [((pl.col(c) < row_mean * 0.5) & (pl.col(c).is_not_null())).cast(pl.Int8) for c in smu_cols]
    df = df.with_columns(pl.sum_horizontal(underperf).alias("str_underperf_count"))

    # Rolling trend of mean string current
    # Inside add_string_health_features, replace the rolling trend section with:

    if "str_mean" in df.columns:
        for label, w in {"3d": 72, "7d": 168}.items():
            expr = pl.col("str_mean").rolling_mean(window_size=w, min_periods=1)
            if has_group:
                expr = expr.over("inverter_idx")
            try:
                df = df.with_columns(expr.alias(f"str_mean_rmean_{label}"))
            except Exception:
                pass

        if "str_mean_rmean_7d" in df.columns:
            df = df.with_columns(
                (pl.col("str_mean") / (pl.col("str_mean_rmean_7d") + 1e-6)).alias("str_trend_7d")
            )

        if "str_worst_ratio" in df.columns:
            expr = pl.col("str_worst_ratio").rolling_mean(window_size=168, min_periods=1)
            if has_group:
                expr = expr.over("inverter_idx")
            try:
                df = df.with_columns(expr.alias("str_worst_ratio_rmean_7d"))
            except Exception:
                pass

    n = len([c for c in df.columns if c.startswith("str_")])
    print(f"    ✅ String health: {n} features")
    return df


def add_grid_features(df: pl.DataFrame) -> pl.DataFrame:
    """Grid quality indicators from meter data."""
    has_group = "inverter_idx" in df.columns
    has_ts = "timestamp" in df.columns and df["timestamp"].dtype in (pl.Datetime,)
    new_cols = []

    # Phase voltage imbalance
    for vr, vy, vb in [("meter_v_r", "meter_v_y", "meter_v_b")]:
        if all(c in df.columns for c in [vr, vy, vb]):
            new_cols.append(
                (pl.max_horizontal([vr, vy, vb]) - pl.min_horizontal([vr, vy, vb]))
                .alias("grid_v_imbalance")
            )
            new_cols.append(
                ((pl.max_horizontal([vr, vy, vb]) - pl.min_horizontal([vr, vy, vb])) /
                 (pl.mean_horizontal([vr, vy, vb]) + 1e-6) * 100)
                .alias("grid_v_imbalance_pct")
            )

    # Phase power imbalance
    for pr, py, pb in [("meter_p_r", "meter_p_y", "meter_p_b")]:
        if all(c in df.columns for c in [pr, py, pb]):
            new_cols.append(
                (pl.max_horizontal([pr, py, pb]) - pl.min_horizontal([pr, py, pb]))
                .alias("grid_p_imbalance")
            )

    # Power factor deviation from 1.0
    for pf_col in ["meter_pf"]:
        if pf_col in df.columns:
            new_cols.append((1.0 - pl.col(pf_col).abs()).alias("grid_pf_dev"))

    # Frequency deviation from 50 Hz
    for freq_col in ["meter_freq", "inv_freq"]:
        if freq_col in df.columns:
            new_cols.append((pl.col(freq_col) - 50.0).abs().alias("grid_freq_dev"))
            break

    if new_cols:
        df = df.with_columns(new_cols)

    # Rolling volatility of grid features
    grid_cols = [c for c in df.columns if c.startswith("grid_")]
    if grid_cols:
        for gc_col in grid_cols:
            expr = pl.col(gc_col).rolling_std(window_size=72, min_periods=2)
            if has_group:
                expr = expr.over("inverter_idx")
            try:
                df = df.with_columns(expr.alias(f"{gc_col}_vol_3d"))
            except Exception:
                pass

    n = len([c for c in df.columns if c.startswith("grid_")])
    print(f"    ✅ Grid features: {n}")
    return df


def add_environmental_features(df: pl.DataFrame) -> pl.DataFrame:
    """Environmental context from ambient temperature sensor."""
    has_group = "inverter_idx" in df.columns
    has_ts = "timestamp" in df.columns and df["timestamp"].dtype in (pl.Datetime,)

    ambient = None
    for c in ["sensor_ambient_temp"]:
        if c in df.columns:
            ambient = c
            break
    if not ambient:
        print("    ⚠️  No ambient_temp — skipping")
        return df

    new_cols = []

    if "inv_temp" in df.columns:
        new_cols.append((pl.col("inv_temp") - pl.col(ambient)).alias("env_temp_delta"))
        new_cols.append((pl.col("inv_temp") / (pl.col(ambient) + 1e-6)).alias("env_temp_ratio"))

    if "inv_power" in df.columns:
        new_cols.append((pl.col("inv_power") / (pl.col(ambient) + 1e-6)).alias("env_power_per_temp"))

    if new_cols:
        df = df.with_columns(new_cols)

    if ambient:
        for label, w in {"3d": 72, "7d": 168}.items():
            expr = pl.col(ambient).rolling_mean(window_size=w, min_periods=1)
            if has_group:
                expr = expr.over("inverter_idx")
            try:
                df = df.with_columns(expr.alias(f"env_ambient_rmean_{label}"))
            except Exception:
                pass

    n = len([c for c in df.columns if c.startswith("env_")])
    print(f"    ✅ Environmental: {n} features")
    return df


PHASE 2.2: ADVANCED FEATURE ENGINEERING

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_1 — 181,958 × 41
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_2 — 80,166 × 74
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_3 — 16,501 × 47
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [13]:
print("=" * 70)
print("PHASE 2.2: ADVANCED FEATURE ENGINEERING")
print("=" * 70)

for plant_name, df in plant_dfs.items():
    print(f"\n{'━'*60}")
    print(f"📂 {plant_name} — {df.height:,} × {df.width}")
    print(f"{'━'*60}")

    w0 = df.width

    df = add_temporal_features(df)
    df = add_rolling_telemetry_features(df)
    print(df.columns)
    df = add_degradation_features(df)
    df = add_inter_inverter_features(df)
    df = add_cumulative_stress_features(df)
    df = add_nighttime_anomaly_features(df)
    df = add_string_health_features(df)
    df = add_grid_features(df)
    df = add_environmental_features(df)

    print(f"\n  📊 {w0} → {df.width} cols (+{df.width - w0} features)")

    plant_dfs[plant_name] = df
    gc.collect()

print("\n✅ All features engineered")

PHASE 2.2: ADVANCED FEATURE ENGINEERING

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_1 — 181,958 × 41
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ✅ Rolling telemetry: 32 features
['inverter_idx', 'timestamp', 'inv_op_state', 'inv_v_ab', 'inv_power', 'inv_v_ca', 'inv_kwh_today', 'inv_freq', 'inv_kwh_total', 'inv_v_bc', 'inv_temp', 'inv_pv1_power', 'smu_string18', 'smu_string10', 'smu_string17', 'smu_string21', 'smu_string4', 'smu_string13', 'smu_string24', 'smu_string2', 'smu_string23', 'smu_string16', 'smu_string14', 'smu_string22', 'smu_string15', 'smu_string6', 'smu_string11', 'smu_string20', 'smu_string19', 'smu_string12', 'smu_string7', 'smu_string1', 'smu_string8', 'smu_string3', 'smu_string5', 'smu_string9', 'hour', 'is_failure_event', 'days_to_next_failure', 'target', 'target_any_within_10d', 'day_of_week', 'month', 'is_daytime', 'roll_power_mean_3d', 'roll_power_std_3d', 'roll_power_mean_7d', 'roll_power_std_7d', 'roll_temp_mean_3d'

### 2.3 Final Cleanup & Export

In [15]:
print("=" * 70)
print("FINAL CLEANUP & EXPORT")
print("=" * 70)

for plant_name, df in plant_dfs.items():
    print(f"\n{'━'*60}")
    print(f"📂 {plant_name}")

    # Replace infinities with null
    float_cols = [c for c in df.columns if df[c].dtype in (pl.Float64, pl.Float32)]
    for c in float_cols:
        df = df.with_columns(
            pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        )

    # Drop internal temp columns
    internal = [c for c in df.columns if c.startswith("_")]
    if internal:
        df = df.drop(internal)

    # Target stats
    if "target" in df.columns:
        n_pos = df["target"].sum()
        n_neg = df.height - n_pos
        ratio = n_neg / max(n_pos, 1)
        print(f"  Target: {n_pos:,} pos / {n_neg:,} neg (1:{ratio:.0f})")

    # Feature breakdown
    cats = {
        "Rolling telemetry": "roll_",
        "Degradation": "degrad_",
        "Inter-inverter": "peer_",
        "Cumulative stress": "stress_",
        "Nighttime anomaly": "anom_",
        "String health (SMU)": "str_",
        "Grid/meter": "grid_",
        "Environmental": "env_",
    }
    print(f"\n  Feature breakdown:")
    for label, prefix in cats.items():
        n = len([c for c in df.columns if c.startswith(prefix)])
        if n > 0:
            print(f"    {label}: {n}")
    print(f"    Total columns: {df.width}")

    # Add plant identifier
    df = df.with_columns(pl.lit(plant_name).alias("plant_id"))

    # Save
    out_path = OUTPUT_DIR / f"{plant_name}_train_ready.parquet"
    df.write_parquet(out_path, compression="zstd", compression_level=3)
    size_mb = out_path.stat().st_size / (1024 * 1024)
    print(f"\n  💾 {out_path.name}: {size_mb:.1f} MB ({df.height:,} × {df.width})")

    plant_dfs[plant_name] = df

print(f"\n✅ All saved to {OUTPUT_DIR}/")

FINAL CLEANUP & EXPORT

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_1
  Target: 11,699 pos / 170,259 neg (1:15)

  Feature breakdown:
    Rolling telemetry: 32
    Degradation: 6
    Inter-inverter: 6
    Cumulative stress: 5
    Nighttime anomaly: 4
    String health (SMU): 10
    Grid/meter: 2
    Total columns: 109

  💾 plant_1_train_ready.parquet: 58.8 MB (181,958 × 110)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_2
  Target: 5,013 pos / 75,153 neg (1:15)

  Feature breakdown:
    Rolling telemetry: 16
    Degradation: 6
    Inter-inverter: 6
    Cumulative stress: 5
    Nighttime anomaly: 4
    String health (SMU): 10
    Grid/meter: 10
    Total columns: 134

  💾 plant_2_train_ready.parquet: 21.2 MB (80,166 × 135)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_3
  Target: 626 pos / 15,875 neg (1:25)

  Feature breakdown:
    Rolling telemetry: 16
    Degradation: 6
    Cumulative stress: 5
    Nighttime ano

### 2.4 Sanity Checks

In [16]:
for plant_name, df in plant_dfs.items():
    print(f"\n{'━'*50}")
    print(f"📂 {plant_name}")

    # Verify op_state is integer-like (not averaged)
    if "inv_op_state" in df.columns:
        sample = df["inv_op_state"].drop_nulls().head(10).to_list()
        print(f"  op_state sample: {sample}")

    # Check target overlap with failure events
    if "target" in df.columns and "is_failure_event" in df.columns:
        both = ((df["target"] == 1) & (df["is_failure_event"] == 1)).sum()
        t1 = df["target"].sum()
        if t1 > 0:
            print(f"  target=1 rows that are ALSO failure events: {both}/{t1} ({both/t1*100:.1f}%)")
            print(f"  (Low % is good — means target is truly forward-looking)")

    # Show positive samples
    if "target" in df.columns:
        key_cols = [c for c in ["timestamp", "inverter_idx", "target",
                                "days_to_next_failure", "is_failure_event",
                                "inv_power", "inv_op_state", "inv_alarm_code"]
                    if c in df.columns]
        pos = df.filter(pl.col("target") == 1).select(key_cols).head(5)
        if pos.height > 0:
            print(f"  Sample positive cases:\n{pos}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_1
  op_state sample: [0.0, 0.0, 0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
  target=1 rows that are ALSO failure events: 0/11699 (0.0%)
  (Low % is good — means target is truly forward-looking)
  Sample positive cases:
shape: (5, 7)
┌───────────────┬──────────────┬────────┬───────────────┬──────────────┬────────────┬──────────────┐
│ timestamp     ┆ inverter_idx ┆ target ┆ days_to_next_ ┆ is_failure_e ┆ inv_power  ┆ inv_op_state │
│ ---           ┆ ---          ┆ ---    ┆ failure       ┆ vent         ┆ ---        ┆ ---          │
│ datetime[ms]  ┆ i16          ┆ i8     ┆ ---           ┆ ---          ┆ f64        ┆ f64          │
│               ┆              ┆        ┆ f64           ┆ i8           ┆            ┆              │
╞═══════════════╪══════════════╪════════╪═══════════════╪══════════════╪════════════╪══════════════╡
│ 2024-03-10    ┆ 0            ┆ 1      ┆ 10.0          ┆ 0            ┆ 1.096667   ┆ 0.0        

## 📝 Next Steps
#
The train-ready parquet files are at `/kaggle/working/train_ready/`
#
```python
# Load for training:
df = pl.read_parquet("/kaggle/working/train_ready/plant_1_train_ready.parquet")
#
# Columns to EXCLUDE from features:
DROP_FOR_TRAINING = [
    "target", "target_any_within_10d", "is_failure_event",
    "days_to_next_failure", "timestamp", "inverter_idx",
    "source_file", "plant_id", "inv_op_state", "inv_alarm_code",
    "hour",  # already captured in is_daytime
]
#
feature_cols = [c for c in df.columns
                if c not in DROP_FOR_TRAINING
                and df[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8)]
#
X = df.select(feature_cols).to_pandas()
y = df["target"].to_pandas()
#
# Handle class imbalance:
scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
#
import xgboost as xgb
model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=500,
    eval_metric="aucpr",
)
```

In [18]:
for plant_name, df in plant_dfs.items():
    n_pos = df["target"].sum()
    n_fail = df["is_failure_event"].sum()
    n_pos_10d = df["target_any_within_10d"].sum()
    print(f"📂 {plant_name}: failure_events={n_fail:,}  target(7-10d)={n_pos:,}  target(0-10d)={n_pos_10d:,}  total={df.height:,}")

📂 plant_1: failure_events=3,087  target(7-10d)=11,699  target(0-10d)=119,505  total=181,958
📂 plant_2: failure_events=2,631  target(7-10d)=5,013  target(0-10d)=64,230  total=80,166
📂 plant_3: failure_events=528  target(7-10d)=626  target(0-10d)=12,377  total=16,501


In [17]:
# Deep dive into failure patterns
for plant_name, df in plant_dfs.items():
    print(f"\n{'━'*50}")
    print(f"📂 {plant_name}")
    
    if "inv_op_state" in df.columns:
        # Distribution of op_state during DAYTIME only
        daytime = df.filter((pl.col("hour") >= 6) & (pl.col("hour") <= 18))
        vc = daytime.group_by("inv_op_state").agg(pl.len().alias("count")).sort("count", descending=True).head(10)
        print(f"\n  Daytime op_state distribution:")
        print(vc)
    
    if "inv_alarm_code" in df.columns:
        daytime = df.filter((pl.col("hour") >= 6) & (pl.col("hour") <= 18))
        vc = daytime.group_by("inv_alarm_code").agg(pl.len().alias("count")).sort("count", descending=True).head(15)
        print(f"\n  Daytime alarm_code distribution:")
        print(vc)
    
    # Check: when power is 0 during daytime, what's the op_state?
    if "inv_power" in df.columns:
        daytime_zero = df.filter(
            (pl.col("hour") >= 6) & (pl.col("hour") <= 18) & (pl.col("inv_power").abs() < 0.1)
        )
        print(f"\n  Daytime zero-power rows: {daytime_zero.height:,} / {df.filter((pl.col('hour') >= 6) & (pl.col('hour') <= 18)).height:,}")
        if "inv_op_state" in daytime_zero.columns:
            vc = daytime_zero.group_by("inv_op_state").agg(pl.len().alias("count")).sort("count", descending=True).head(5)
            print(f"  Op_state when daytime zero-power:")
            print(vc)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_1

  Daytime op_state distribution:
shape: (2, 2)
┌──────────────┬───────┐
│ inv_op_state ┆ count │
│ ---          ┆ ---   │
│ f64          ┆ u32   │
╞══════════════╪═══════╡
│ -1.0         ┆ 95820 │
│ 0.0          ┆ 7801  │
└──────────────┴───────┘

  Daytime zero-power rows: 3,087 / 103,621
  Op_state when daytime zero-power:
shape: (2, 2)
┌──────────────┬───────┐
│ inv_op_state ┆ count │
│ ---          ┆ ---   │
│ f64          ┆ u32   │
╞══════════════╪═══════╡
│ 0.0          ┆ 3017  │
│ -1.0         ┆ 70    │
└──────────────┴───────┘

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 plant_2

  Daytime op_state distribution:
shape: (9, 2)
┌──────────────┬───────┐
│ inv_op_state ┆ count │
│ ---          ┆ ---   │
│ i64          ┆ u32   │
╞══════════════╪═══════╡
│ 0            ┆ 39490 │
│ 37120        ┆ 2929  │
│ 5120         ┆ 2770  │
│ 4864         ┆ 186   │
│ 5632         ┆ 130   │
│ 21760        ┆ 9     │
│ 33280    

In [ ]:
!rm -rf /kaggle/working/*


In [19]:
df.groupby("plant")["is_failure_event"].sum()

AttributeError: 'DataFrame' object has no attribute 'groupby'

In [22]:
import polars as pl

df = pl.read_parquet("/kaggle/working/train_ready/plant_3_train_ready.parquet")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nSchema:")
print(df.schema)

print("\nTarget distribution:")
print(df["target"].value_counts())

print("\nFailure events:")
print(df["is_failure_event"].sum())

print("\nSample rows:")
print(df.head(5))

Shape: (16501, 102)

Columns:
['inverter_idx', 'timestamp', 'inv_alarm_code', 'inv_op_state', 'inv_pv1_current', 'inv_pv1_power', 'inv_pv1_voltage', 'inv_kwh_midnight', 'inv_kwh_total', 'inv_kwh_today', 'inv_power', 'meter_base_meter_kwh_import', 'meter_base_meter_kwh_total', 'meter_original_meter_kwh_import', 'meter_original_meter_kwh_total', 'meter_pf', 'meter_freq', 'meter_p_b', 'meter_p_y', 'meter_p_r', 'meter_v_b', 'meter_v_y', 'meter_v_r', 'meter_meter_active_power', 'meter_meter_kwh_import', 'meter_meter_kwh_total', 'inv_temp', 'smu_string6', 'smu_string11', 'smu_string2', 'smu_string10', 'smu_string7', 'smu_string1', 'smu_string8', 'smu_string4', 'smu_string3', 'smu_string5', 'smu_string9', 'inv_limit_percent', 'meter_meter_reactive_power', 'meter_meter_apparent_power', 'smu_string12', 'hour', 'day_of_week', 'month', 'is_daytime', 'roll_power_mean_3d', 'roll_power_std_3d', 'roll_power_mean_7d', 'roll_power_std_7d', 'roll_temp_mean_3d', 'roll_temp_std_3d', 'roll_temp_mean_7d', '